In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import random
from tqdm import tqdm

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [6]:
num_letters = 5

# Cria um dicionário que mapeia pares de letras como "AB", "DE" para inteiros únicos
tok2idx = {f"{chr(65 + i)}{chr(65 + j)}": i * num_letters + j for i in range(num_letters) for j in range(num_letters)}
print(tok2idx)

# Tokeniza um texto
def tokenizer(text):
    words = text.split()
    output = torch.tensor([tok2idx[word] for word in words], dtype=torch.long)
    return output

# Função para converter um tensor de inteiros em texto
def get_text_from_tensor(tensor):
    return ' '.join([list(tok2idx.keys())[i] for i in tensor])

{'AA': 0, 'AB': 1, 'AC': 2, 'AD': 3, 'AE': 4, 'BA': 5, 'BB': 6, 'BC': 7, 'BD': 8, 'BE': 9, 'CA': 10, 'CB': 11, 'CC': 12, 'CD': 13, 'CE': 14, 'DA': 15, 'DB': 16, 'DC': 17, 'DD': 18, 'DE': 19, 'EA': 20, 'EB': 21, 'EC': 22, 'ED': 23, 'EE': 24}


📌 Objetivo:
Criar um dicionário (dict) que mapeia todos os pares possíveis de letras (bigramas) entre 'A' e 'E' (5 letras), para um número inteiro único entre 0 e 24.

🧠 Parte 1: Entendendo chr(65 + i) e chr(65 + j)
chr() converte um número em caractere ASCII.

O código ASCII de 'A' é 65.

Então:

chr(65 + 0) → 'A'

chr(65 + 1) → 'B'

chr(65 + 2) → 'C'

chr(65 + 3) → 'D'

chr(65 + 4) → 'E'

Com num_letters = 5, os loops vão gerar todas as 25 combinações possíveis de 2 letras, de 'AA' até 'EE'.


🧩 Parte 2: O dicionário por compreensão (dict comprehension)



```
tok2idx = {}
for i in range(num_letters):         # i de 0 a 4
    for j in range(num_letters):     # j de 0 a 4
        key = f"{chr(65 + i)}{chr(65 + j)}"   # Ex: "AB"
        value = i * num_letters + j          # Ex: 1*5 + 0 = 5
        tok2idx[key] = value
```
Então, para cada par (i, j), ele cria uma string como chave (ex: 'BD') e um número como valor, garantindo que:

Cada par de letras seja único

Cada valor também seja único, de 0 até
5 x 5- 1 = 24

✅ Resultado Final
Um dicionário com 25 pares de letras como chave e seus índices únicos como valor:



```
{
  'AA': 0, 'AB': 1, ..., 'AE': 4,
  'BA': 5, ..., 'BE': 9,
  ...
  'EE': 24
}
```
Função tokenizer(text):

```
def tokenizer(text):
    words = text.split()
    output = torch.tensor([tok2idx[word] for word in words], dtype=torch.long)
    return output
```
Divide o texto em palavras (espera que cada palavra seja um par de letras, como "AB", "CD").

Converte cada par para o número correspondente usando tok2idx.

Retorna um torch.Tensor de inteiros.

```
tokenizer("AB BE CA")
# Saída: tensor([1, 9, 10])
```

```
def get_text_from_tensor(tensor):
    return ' '.join([list(tok2idx.keys())[i] for i in tensor])
```
Faz o inverso: pega cada índice e retorna a chave (par de letras) correspondente.

Usa list(tok2idx.keys())[i] para fazer o mapeamento inverso.



```
get_text_from_tensor(torch.tensor([1, 9, 10]))
# Saída: "AB BE CA"
```







In [7]:
# Gera exemplos de treinamento
def generate_examples(N):
    examples = []
    max_letter = 65 + num_letters - 1
    for _ in range(N):
        # Gera dois pares de letras aleatórios
        first_pair = f"{chr(random.randint(65, max_letter))}{chr(random.randint(65, max_letter))}"  # Random pair like "AB"
        second_pair = f"{chr(random.randint(65, max_letter))}{chr(random.randint(65, max_letter))}"  # Random pair like "DE"

        # Gera o target que é a primeira letra do primeiro par e a segunda letra do segundo par
        target = f"{first_pair[0]}{second_pair[1]}"

        # Adiciona o exemplo à lista
        examples.append((first_pair, second_pair, target))

    return examples

In [8]:
N = 5000
examples = generate_examples(N)

In [11]:
print(examples[100])

('AA', 'BA', 'AA')


In [13]:
# Cria um dataset com os exemplos
class SimpleTokenDataset(Dataset):
    def __init__(self, examples):
        self.data = examples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_seq1, input_seq2, target_seq = self.data[idx] # Pega o exemplo no índice idx
        src = tokenizer(input_seq1 + " " + input_seq2) # Concatena os dois pares de entrada com espaço e tokeniza -> src = tensor([8, 9])   # BD = 8, AE = 9
        tgt = tokenizer(target_seq).squeeze() # Tokeniza o par-alvo: tgt = tensor(9)        # BE = 9
        return src, tgt

In [14]:
dataset = SimpleTokenDataset(examples)

num_val = 0.2 * N # exemplos vão para o conjunto de validação.

train_set = Subset(dataset, range(N - int(num_val))) # pega os 4000 primeiros valores
val_set = Subset(dataset, range(N - int(num_val), N)) # pega os 1000 últimos valores

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)

In [15]:
class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)

        # Apenas o último hidden state é utilizado
        _, (h_n, _) = self.lstm(x)

        out = self.fc(h_n.squeeze(0))
        return out


vocab_size = 10 * 10
embedding_dim = 128
hidden_dim = 256
output_dim = vocab_size

model = TextGenerator(vocab_size, embedding_dim, hidden_dim, output_dim)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
epochs = 5
for epoch in range(epochs):
    # Treinamento
    model.train()
    train_loss = 0
    for input_seq, target_seq in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        output = model(input_seq)
        loss = criterion(output, target_seq)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Train Loss: {avg_train_loss:.4f}')

    # Validação
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for input_seq, target_seq in val_loader:
            output = model(input_seq)
            loss = criterion(output, target_seq)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f'Val Loss: {avg_val_loss:.4f}')

Train Epoch 1/5: 100%|██████████| 250/250 [00:02<00:00, 95.76it/s] 


Train Loss: 1.0788
Val Loss: 0.0301


Train Epoch 2/5: 100%|██████████| 250/250 [00:01<00:00, 127.15it/s]


Train Loss: 0.0148
Val Loss: 0.0082


Train Epoch 3/5: 100%|██████████| 250/250 [00:02<00:00, 100.74it/s]


Train Loss: 0.0055
Val Loss: 0.0040


Train Epoch 4/5: 100%|██████████| 250/250 [00:02<00:00, 108.48it/s]


Train Loss: 0.0030
Val Loss: 0.0024


Train Epoch 5/5: 100%|██████████| 250/250 [00:02<00:00, 124.31it/s]


Train Loss: 0.0019
Val Loss: 0.0016


In [18]:
# Inferência
input_text = "DE AE"
inputs = tokenizer(input_text)

output = model(inputs)
predicted_token = output.argmax().unsqueeze(0)
predicted_text = get_text_from_tensor(predicted_token)

print(f'Input: {input_text}, Predicted: {predicted_text}')

Input: DE AE, Predicted: DE


In [19]:
def generate_autoregressive_text(model, input_text, seq_len):
    inputs = tokenizer(input_text)
    outputs = []

    for _ in range(seq_len):
        output = model(inputs)
        predicted_token = output.argmax().unsqueeze(0)
        inputs = torch.cat([inputs[-1].unsqueeze(0), predicted_token])
        outputs.append(predicted_token)

    return get_text_from_tensor(outputs)

input_text = "AB CD"
seq_len = 5
predicted_text = generate_autoregressive_text(model, input_text, seq_len)

print(f'Input: {input_text}, Predicted: {predicted_text}')

Input: AB CD, Predicted: AD CD AD CD AD
